# MSTCN-SE Reproduction on Kaggle

Notebook chạy nhanh để tái hiện Multi-Scale TCN cho speech enhancement. Bật GPU T4 trước khi chạy.

In [ ]:
import os, sys, glob
print('Python OK')
print('Input folders:')
for p in glob.glob('/kaggle/input/*'):
    print(' -', p)

## 1. Cài thư viện

In [ ]:
!pip -q install librosa soundfile tqdm matplotlib

## 2. Kiểm tra code project

In [ ]:
import os
PROJECT = '/kaggle/working/MSTCN_Kaggle'
# Nếu bạn upload zip này qua nút Upload, Kaggle thường đặt nó trong /kaggle/input hoặc /kaggle/working.
# Cell này tự tìm và giải nén nếu cần.
if not os.path.exists(PROJECT):
    candidates = glob.glob('/kaggle/input/**/*.zip', recursive=True) + glob.glob('/kaggle/working/*.zip')
    print('Zip candidates:', candidates)
    for z in candidates:
        if 'MSTCN_Kaggle' in z or 'mstcn' in z.lower():
            !unzip -q -o "{z}" -d /kaggle/working/
            break
os.chdir(PROJECT if os.path.exists(PROJECT) else '/kaggle/working')
print('CWD =', os.getcwd())
!ls -la

## 3. Tự dò thư mục dữ liệu wav

Bạn có thể chỉnh lại các đường dẫn bên dưới cho đúng dataset của bạn.

In [ ]:
from pathlib import Path
import glob, os

def count_wavs(p):
    return len(glob.glob(str(Path(p) / '**/*.wav'), recursive=True)) if p else 0

for root in glob.glob('/kaggle/input/*'):
    print('
ROOT:', root)
    for d, subdirs, files in os.walk(root):
        n = sum(1 for f in files if f.lower().endswith('.wav'))
        if n > 0:
            print(' ', d, 'wav=', n)

## 4. Cấu hình đường dẫn

Có 2 chế độ:

- `paired`: có thư mục clean và noisy tương ứng. Phù hợp dataset kiểu VCTK noisy/clean.
- `mix`: có clean speech và noise riêng, code tự trộn nhiễu theo SNR.

In [ ]:
# ===== SỬA 4 ĐƯỜNG DẪN NÀY CHO ĐÚNG DATASET CỦA BẠN =====
MODE = 'paired'  # 'paired' hoặc 'mix'

# Ví dụ nếu dataset VCTK có clean/noisy train/test, hãy trỏ đúng thư mục chứa wav.
CLEAN_TRAIN = '/kaggle/input/VCTK-NoiseX92/clean_trainset_28spk_wav'
NOISY_TRAIN = '/kaggle/input/VCTK-NoiseX92/noisy_trainset_28spk_wav'
CLEAN_VAL   = '/kaggle/input/VCTK-NoiseX92/clean_testset_wav'
NOISY_VAL   = '/kaggle/input/VCTK-NoiseX92/noisy_testset_wav'

# Nếu dùng MODE='mix' thì dùng 4 biến này:
NOISE_TRAIN = '/kaggle/input/noisex92/noise_train'
NOISE_VAL   = '/kaggle/input/noisex92/noise_val'

for name, p in [('CLEAN_TRAIN',CLEAN_TRAIN),('NOISY_TRAIN',NOISY_TRAIN),('CLEAN_VAL',CLEAN_VAL),('NOISY_VAL',NOISY_VAL),('NOISE_TRAIN',NOISE_TRAIN),('NOISE_VAL',NOISE_VAL)]:
    print(name, p, 'wav=', count_wavs(p))

## 5. Train nhanh thử nghiệm

Để test trước, đặt `MAX_ITEMS=100` và `EPOCHS=2`. Khi chạy thật thì tăng lên.

In [ ]:
EPOCHS = 2
BATCH = 4
MAX_ITEMS = 100  # đổi None để train toàn bộ
OUT = '/kaggle/working/runs/mstcn_se2'

if MODE == 'paired':
    cmd = f"python train.py --mode paired --clean_train '{CLEAN_TRAIN}' --clean_val '{CLEAN_VAL}' --noisy_train '{NOISY_TRAIN}' --noisy_val '{NOISY_VAL}' --epochs {EPOCHS} --batch {BATCH} --max_items {MAX_ITEMS} --out {OUT}"
else:
    cmd = f"python train.py --mode mix --clean_train '{CLEAN_TRAIN}' --clean_val '{CLEAN_VAL}' --noise_train '{NOISE_TRAIN}' --noise_val '{NOISE_VAL}' --epochs {EPOCHS} --batch {BATCH} --max_items {MAX_ITEMS} --out {OUT}"
print(cmd)
!{cmd}

## 6. Vẽ loss

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
hist = pd.read_csv('/kaggle/working/runs/mstcn_se2/history.csv')
display(hist)
plt.figure(figsize=(6,4))
plt.plot(hist['epoch'], hist['train_loss'], label='train')
plt.plot(hist['epoch'], hist['val_loss'], label='val')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.grid(True)
plt.show()

## 7. Enhance thử một file noisy

In [ ]:
import glob, os
noisy_files = glob.glob(NOISY_VAL + '/**/*.wav', recursive=True)
print('Noisy files:', len(noisy_files))
if noisy_files:
    test_wav = noisy_files[0]
    print('Test:', test_wav)
    !python inference.py --ckpt /kaggle/working/runs/mstcn_se2/best.pt --noisy_wav "{test_wav}" --out_wav /kaggle/working/enhanced.wav

## 8. Nghe audio trong notebook

In [ ]:
from IPython.display import Audio, display
if noisy_files:
    print('Noisy:')
    display(Audio(test_wav))
    print('Enhanced:')
    display(Audio('/kaggle/working/enhanced.wav'))

## Ghi chú

Muốn chạy giống bài báo hơn: dùng TIMIT làm clean speech và NOISEX-92 làm noise, chia noise 60%/20%/20%, SNR từ -5 đến 15 dB, train nhiều epoch hơn.